In [15]:
!pip install -q transformers datasets librosa soundfile kagglehub

In [16]:
import os
import torch
import librosa
import numpy as np  # ✅ ADD THIS
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import Wav2Vec2Processor, Wav2Vec2Model

In [17]:
import kagglehub

# REAL DATA
real_path = kagglehub.dataset_download("mathurinache/the-lj-speech-dataset")
print("Real dataset path:", real_path)

# FAKE DATA
fake_path = kagglehub.dataset_download("andreadiubaldo/wavefake-test")
print("Fake dataset path:", fake_path)

Using Colab cache for faster access to the 'the-lj-speech-dataset' dataset.
Real dataset path: /kaggle/input/the-lj-speech-dataset
Fake dataset path: /root/.cache/kagglehub/datasets/andreadiubaldo/wavefake-test/versions/1


In [18]:
import os

real_files = []
fake_files = []

# Collect REAL audio
for root, dirs, files in os.walk(real_path):
    for file in files:
        if file.endswith(".wav"):
            real_files.append(os.path.join(root, file))

# Collect FAKE audio
for root, dirs, files in os.walk(fake_path):
    for file in files:
        if file.endswith(".wav"):
            fake_files.append(os.path.join(root, file))

print("Total REAL:", len(real_files))
print("Total FAKE:", len(fake_files))

Total REAL: 13100
Total FAKE: 134266


In [19]:
min_len = min(len(real_files), len(fake_files))

real_files = real_files[:min_len]
fake_files = fake_files[:min_len]

all_files = real_files + fake_files
labels = [0]*min_len + [1]*min_len  # 0=REAL, 1=FAKE

print("Final dataset size:", len(all_files))

Final dataset size: 26200


In [20]:
from sklearn.model_selection import train_test_split

train_files, val_files, train_labels, val_labels = train_test_split(
    all_files, labels, test_size=0.2, random_state=42
)

In [21]:
import torch
from transformers import Wav2Vec2Processor, Wav2Vec2Model
import torch.nn as nn

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

processor = Wav2Vec2Processor.from_pretrained("facebook/wav2vec2-base")

In [22]:
import librosa
from torch.utils.data import Dataset

class AudioDataset(Dataset):
    def __init__(self, files, labels):
        self.files = files
        self.labels = labels
        self.max_len = 5 * 16000  # 5 seconds

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        audio, sr = librosa.load(self.files[idx], sr=16000)

        # Cut if too long
        if len(audio) > self.max_len:
            audio = audio[:self.max_len]

        # Pad if too short
        if len(audio) < self.max_len:
            padding = self.max_len - len(audio)
            audio = np.pad(audio, (0, padding))

        inputs = processor(
            audio,
            sampling_rate=16000,
            return_tensors="pt",
            return_attention_mask=True
        )

        return {
            "input_values": inputs["input_values"].squeeze(0),
            "attention_mask": inputs["attention_mask"].squeeze(0),
            "label": torch.tensor(self.labels[idx])
        }

In [23]:
class RealFakeDetector(nn.Module):
    def __init__(self):
        super().__init__()
        self.wav2vec = Wav2Vec2Model.from_pretrained("facebook/wav2vec2-base")

        # 🔥 Freeze Wav2Vec2
        for param in self.wav2vec.parameters():
            param.requires_grad = False

        self.classifier = nn.Linear(768, 2)

    def forward(self, input_values, attention_mask):
        with torch.no_grad():
            outputs = self.wav2vec(input_values, attention_mask=attention_mask)

        hidden = outputs.last_hidden_state.mean(dim=1)
        return self.classifier(hidden)

In [24]:
from torch.utils.data import DataLoader

train_dataset = AudioDataset(train_files, train_labels)
val_dataset = AudioDataset(val_files, val_labels)

train_loader = DataLoader(train_dataset, batch_size=4, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=4)

model = RealFakeDetector().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-5)

Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_hid.bias             | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
project_hid.weight           | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [25]:
for epoch in range(3):
    model.train()
    total_loss = 0

    for batch in train_loader:
        optimizer.zero_grad()

        input_values = batch["input_values"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["label"].to(device)

        outputs = model(input_values, attention_mask)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} Loss:", total_loss/len(train_loader))

torch.save(model.state_dict(), "real_fake_model.pth")

Epoch 1 Loss: 0.6919804112374327
Epoch 2 Loss: 0.6877920121409511
Epoch 3 Loss: 0.6832760644436793


In [26]:
def predict_audio(file_path):
    audio, sr = librosa.load(file_path, sr=16000)

    inputs = processor(
        audio,
        sampling_rate=16000,
        return_tensors="pt",
        padding=True,
        return_attention_mask=True
    )

    input_values = inputs["input_values"].to(device)
    attention_mask = inputs["attention_mask"].to(device)

    model.eval()
    with torch.no_grad():
        outputs = model(input_values, attention_mask)
        probs = torch.softmax(outputs, dim=1)
        confidence, pred = torch.max(probs, dim=1)

    if pred.item() == 0:
        print("Prediction: REAL")
        print("Reason: Natural speech variations detected.")
    else:
        print("Prediction: FAKE")
        print("Reason: Synthetic vocoder artifacts detected.")

    print("Confidence:", round(confidence.item()*100,2), "%")

In [ ]:
predict_audio("/content/0.wav")

Prediction: REAL
Reason: Natural speech variations detected.
Confidence: 54.21 %
